In [161]:
import pandas as pd
import ntpath
from openpyxl import load_workbook

#============================================================
#1. Data Extraction and Dilution Factor Correction (10X)
#============================================================

# 1. Extract raw GC data
def extract_GC_data(input_file, output_sheet_name):
    # Read the Excel file 
    xls = pd.ExcelFile(input_files[0], engine='openpyxl')
    sheet_names = xls.sheet_names
    sheet_names = [sheet for sheet in sheet_names if sheet != output_sheet_name]
    
    # Dictionary to store the data
    output_data = {
        "sampleID": [],
        "Butanol": [],
        "Acetic Acid": [],
        "Butyric Acid": [],
        "Hexanoic Acid": [],
    }

    # Product
    products = {
        "Butanol",
        "Acetic Acid",
        "Butyric Acid",
        "Hexanoic Acid"
    }
    
    # Scan for product headers
    for sheet in sheet_names:
#        print(f"Checking sheet: {sheet}")
        df = pd.read_excel(input_file, sheet_name=sheet, engine="openpyxl", header=None)
        if df.empty:
            continue
#        print(f"DataFrame shape: {df.shape}")
#        print(len(sheet_names))
#        display(df)


        
        # Start to scan
        i = 0
#        print(f"conc_results: {conc_results}")
#        print(f"Dataframe size is: {len(df)}")
        for i in range(len(df)):
            if pd.notna(df.iloc[i,0]) == True:
                sample_id = str(df.iloc[i, 0]).strip()                       # sampleID in A column
                sample_id = sample_id[:-4]                                   # Remove automatic sample numbering
                output_data["sampleID"].append(sample_id)                    # Add the sampleID to the dictionary
#                print(f"Initial: output_data = {output_data}")
#                print(f"Initial: key = {key}, i ={i}")
                for key in products:
                    output_data[key].append(0)
#                    print(f"Initialize output_data with 0: output_data = {output_data}")
#                    print(f"Initialize output_data with 0: key = {key}, i ={i}")
            for key in products:
                prod = str(df.iloc[i, 1])                                    # Column B: Product name
                conc = float(df.iloc[i, 3]) * 10                             # Column D: Concentration of product, 10X Dilution factor
                if key == prod:
                    output_data[key][-1] = conc 
#                    print(f"If: output_data = {output_data}")
#                    print(f"If: key = {key}, i ={i}")
                else:
 #                   print(f"Else; output_data is:{output_data}")
 #                   print(f"Else; key, and i is: {key}, {i}")
                    continue
                    
    # Convert the dictionary to a DataFrame
    result_df = pd.DataFrame(output_data)
    #display(result_df.head(10))
    

    # Add the result sheet
    with pd.ExcelWriter(input_file, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
         result_df.to_excel(writer, sheet_name=output_sheet_name, index=False)
        
input_files = ["test.xlsx"]
output_sheet_name = "Processed Concentration Data"

for input_file in input_files:
    extract_GC_data(input_file, output_sheet_name)


In [141]:
import re
import math
import ntpath
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

#========================================
#2. sampleID Parsing and Data Processing
#========================================

test_id = "Condition-100Time-1"

def parse_sample_id(sample_id: str):
    sample_id = str(sample_id).strip()
    parts = sample_id.rsplit("-", 2)                  # Split from the right at up to 2 "-" delimiters (max 3 parts)

    condition_name = parts[0]                         # Strain/Condition name
    batch_num = int(parts[2])
    
    time_digits = re.findall(r"\d+", time_part)       # Extract only the number in time_part
    time_val = int(time_digits[0])                    # Use the first number in time_digits
    return condition_name, time_val, batch_num


#parse_sample_id(test_id)    

def process_data(input_file):
    wb = load_workbook(input_file, data_only=True)
    # Processed Concentration Data 시트 읽기
    df_conc = pd.read_excel(input_file, sheet_name="Processed Concentration Data", engine="openpyxl")


    # sampleID 파싱
    df_conc["parsed_cell"] = None
    df_conc["parsed_time"] = None
    df_conc["parsed_num"] = None

    for idx, row in df_conc.iterrows():
        sid = str(row["sampleID"])
        cell_name, time_val, sample_num = parse_sample_id(sid)
        df_conc.loc[idx, "parsed_cell"] = cell_name
        df_conc.loc[idx, "parsed_time"] = time_val
        df_conc.loc[idx, "parsed_num"] = sample_num

    # 정렬
    df_conc_sorted = df_conc.sort_values(
        by=["parsed_cell", "parsed_time", "parsed_num"],
        na_position="last"
    )

    final_rows = []
    skip_next = False
    n = len(df_conc_sorted)

    for i in range(n):
        if skip_next:
            skip_next = False
            continue

        row_i = df_conc_sorted.iloc[i]
        num_i = row_i["parsed_num"]

        if pd.isna(num_i):
            final_rows.append(row_i)
            continue

        num_i = int(num_i)

        # 평균/std 없이 그대로 저장
    df_final = df_conc_sorted.drop(
        columns=["parsed_cell", "parsed_time", "parsed_num"],
        errors="ignore"
    )

    # concentration 시트 생성
    if "concentration" in wb.sheetnames:
        del wb["concentration"]
    ws_conc = wb.create_sheet("concentration")

    for row_data in dataframe_to_rows(df_final, index=False, header=True):
        ws_conc.append(row_data)

    output_file = input_file.replace(".xlsx", "_processed.xlsx")
    wb.save(output_file)
    print(f"Done! Check 'concentration' sheet in '{output_file}'.")



('M-THCT-20mM-1010', 120, 1)

In [ ]:
of M-THCT_GC_10202026
def sample_stdev_2vals(x, y):
    """
    표본표준편차(stdev.s)를 2개 값에 대해 계산
    n = 2 이므로 분모는 (n - 1) = 1
    """
    mean = (x + y) / 2.0
    return math.sqrt(((x - mean) ** 2 + (y - mean) ** 2) / 1.0)


def convert_area_to_conc(area_value, col_name, slope_dict, yaxis_dict):
    """
    (Area - y_axis) / slope × 10.5 계산
    col_name = "Area Glucose" -> key = "Glucose"
    """
    if pd.isna(area_value):
        return None

    key = col_name.replace("Area ", "").strip()
    if key not in slope_dict or key not in yaxis_dict:
        return None

    slope = slope_dict[key]
    y_axis = yaxis_dict[key]

    conc = (area_value - y_axis) / slope
    conc *= 10.5
    return conc


def process_data(input_file):
    wb = load_workbook(input_file, data_only=True)
    # Processed Area Data 시트 읽기
    df_raw = pd.read_excel(input_file, sheet_name="Processed Area Data", engine="openpyxl")

    # 농도 변환용 DataFrame
    df_conc = df_area.copy()

    # "Area XXX" 열 찾기
    area_cols = [c for c in df_conc.columns if c.startswith("Area ")]

    # 농도 변환만 (std 없음)
    for c in area_cols:
        df_conc[c] = df_conc[c].apply(
            lambda val: convert_area_to_conc(val, c, slope_dict, yaxis_dict)
        )

    # sampleID 파싱
    df_conc["parsed_cell"] = None
    df_conc["parsed_time"] = None
    df_conc["parsed_num"] = None

    for idx, row in df_conc.iterrows():
        sid = str(row["sampleID"])
        cell_name, time_val, sample_num = parse_sample_id(sid)
        df_conc.loc[idx, "parsed_cell"] = cell_name
        df_conc.loc[idx, "parsed_time"] = time_val
        df_conc.loc[idx, "parsed_num"] = sample_num

    # 정렬
    df_conc_sorted = df_conc.sort_values(
        by=["parsed_cell", "parsed_time", "parsed_num"],
        na_position="last"
    )

    final_rows = []
    skip_next = False
    n = len(df_conc_sorted)

    for i in range(n):
        if skip_next:
            skip_next = False
            continue

        row_i = df_conc_sorted.iloc[i]
        num_i = row_i["parsed_num"]

        if pd.isna(num_i):
            final_rows.append(row_i)
            continue

        num_i = int(num_i)

        # 평균/std 없이 그대로 저장
    df_final = df_conc_sorted.drop(
        columns=["parsed_cell", "parsed_time", "parsed_num"],
        errors="ignore"
    )

    # concentration 시트 생성
    if "concentration" in wb.sheetnames:
        del wb["concentration"]
    ws_conc = wb.create_sheet("concentration")

    for row_data in dataframe_to_rows(df_final, index=False, header=True):
        ws_conc.append(row_data)

    output_file = input_file.replace(".xlsx", "_processed.xlsx")
    wb.save(output_file)
    print(f"Done! Check 'concentration' sheet in '{output_file}'.")


# 실행 예시
if __name__ == "__main__":
    input_file = "M-THT_05042026.xlsx"
    process_data(input_file)